In [23]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict,Annotated,Literal
from langchain_core.messages import HumanMessage,SystemMessage
import operator

In [24]:
load_dotenv()

generator_llm=ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",
    temperature=0
)
evaluator_llm=ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",
    temperature=0
)
optimizer_llm=ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",
    temperature=0
)

In [25]:
class TweetState(TypedDict):
    topic: str
    tweet: str
    evaluation: Literal['approved','needs_improvement']
    feedback: str
    iteration: int
    max_iterations: int

    tweet_history: Annotated[list[str],operator.add]
    feeedback_history: Annotated[list[str],operator.add]

In [26]:
from pydantic import BaseModel, Field

class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation result.")
    feedback: str = Field(..., description="feedback for the tweet.")

In [27]:
structured_evaluator_llm=evaluator_llm.with_structured_output(TweetEvaluation)

In [28]:
def generate_tweet(state:TweetState):
    # prompt
    messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(content=f"""
Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day english
""")
    ]

    # send to generator_llm
    response=generator_llm.invoke(messages).content

    # response 
    return {'tweet':response,'tweet_history':[response]}

In [29]:
def evaluate_tweet(state:TweetState):
    # prompt
    messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
Evaluate the following tweet:

Tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality - Is this fresh, or have you seen it a hundred times before?  
2. Humor - Did it genuinely make you smile, laugh, or chuckle?  
3. Punchiness - Is it short, sharp, and scroll-stopping?  
4. Virality Potential - Would people retweet or share it?  
5. Format - Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"  
- feedback: One paragraph explaining the strengths and weaknesses 
""")
]

    # send to evaluator_llm
    response=structured_evaluator_llm.invoke(messages)

    return {'evaluation':response.evaluation,'feedback':response.feedback,'feedback_history':[response.feedback]}

In [30]:
def optimize_tweet(state:TweetState):
    # prompt
    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
Improve the tweet based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Tweet:
{state['tweet']}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
""")
    ]

    response=optimizer_llm.invoke(messages).content
    iteration=state['iteration']+1
    return {'tweet':response,'iteration':iteration}

In [31]:
def route_evaluation(state:TweetState):
    if state['evaluation']=='approved' or state['iteration']>=state['max_iterations']:
        return 'approved'
    else:
        return 'needs_improvement'

In [32]:
graph=StateGraph(TweetState)

# nodes
graph.add_node('generate',generate_tweet)
graph.add_node('evaluate',evaluate_tweet)
graph.add_node('optimize',optimize_tweet)

# edges
graph.add_edge(START,'generate')
graph.add_edge('generate','evaluate')
graph.add_conditional_edges('evaluate',route_evaluation,{'approved':END,'need_improvement':'optimize'})
graph.add_edge('optimize','evaluate')

workflow=graph.compile()

In [ ]:
initial_state={
    'topic':'Indian railways',
    'iteration':1,
    'max_iterations':5
}
result=workflow.invoke(initial_state)

In [ ]:
result

{'topic': 'Indian railways',
 'tweet': 'Indian Railways: The only place where I genuinely believe I might finally finish that novel I started in 2005. Punctuality is just a rumour, but the journey is eternal.',
 'evaluation': 'approved',
 'feedback': "This tweet is genuinely humorous and highly relatable, especially for anyone familiar with Indian Railways. The originality comes from framing the notorious delays as an opportunity to finally complete a long-abandoned novel, which is a fresh take on a common observation. The line 'Punctuality is just a rumour, but the journey is eternal' is a sharp, punchy closer that elevates the humor and makes it highly shareable. It adheres perfectly to tweet format, staying well under the character limit and avoiding any setup-punchline or Q&A structures, giving it strong virality potential within its target audience.",
 'iteration': 1,
 'max_iterations': 5,
 'tweet_history': ['Indian Railways: The only place where I genuinely believe I might finall